# 응용 2. 칼만 필터로 다듬은 값으로 밀기

**개념:** [개념 3 칼만 필터](../01_concepts/03_kalman_filter.ipynb)

외삽 식은 응용 1과 같은 **직선**이다. 바뀌는 것은 **외삽에 넣는 값**이다.

```
응용 1:  AIS 보고 원래 값 ─────────────────▶ 직선 외삽
응용 2:  AIS 보고 ──▶ 칼만 필터(다듬기) ──▶ 직선 외삽
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sim

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
KN = sim.KN

## 1. 배에 맞게 상태와 측정 정하기

개념 3은 1차원(위치, 속도)이었다. 배는 2차원이라 상태를 4개로 둔다.

$$\mathbf{x} = [e,\ n,\ v_e,\ v_n] \quad (\text{동·북 위치와 속도})$$

모델은 **등속(Constant Velocity)**, 예측은 선형 외삽이다: $e^- = e + v_e\,dt,\ n^- = n + v_n\,dt$.

AIS 보고 하나에서 측정 4개를 만든다.
- 위치 $e, n$ (±5m)
- SOG·COG를 응용 1처럼 북·동으로 나눈 속도 $v_e = v\sin C,\ v_n = v\cos C$

필터가 다듬은 상태에서 직선으로 민다:

$$e(\tau) = \hat e + \hat v_e\tau, \qquad n(\tau) = \hat n + \hat v_n\tau$$

In [ ]:
def kalman_cv(rep, q=0.05):
    """등속 칼만 필터. 보고마다 다듬은 상태 [e, n, ve, vn] 목록을 돌려준다."""
    sp = sim.POS_NOISE
    H = np.eye(4)
    x = np.array([rep["e"][0], rep["n"][0], 0.0, 0.0])
    P = np.diag([sp**2, sp**2, 10.0, 10.0])
    states = []
    for k in range(len(rep["t"])):
        if k > 0:                                                  # 예측 (선형 외삽)
            dt = rep["t"][k] - rep["t"][k - 1]
            F = np.eye(4); F[0, 2] = F[1, 3] = dt
            G = np.array([[dt**2 / 2, 0], [0, dt**2 / 2], [dt, 0], [0, dt]])
            x = F @ x
            P = F @ P @ F.T + q * G @ G.T
        v, c = rep["sog"][k] * KN, np.radians(rep["cog"][k])       # 보정
        z = np.array([rep["e"][k], rep["n"][k], v * np.sin(c), v * np.cos(c)])
        sv = np.hypot(sim.SOG_NOISE * KN, v * np.radians(sim.COG_NOISE))
        R = np.diag([sp**2, sp**2, sv**2, sv**2])
        K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)
        x = x + K @ (z - H @ x)
        P = (np.eye(4) - K @ H) @ P
        states.append(x.copy())
    return np.array(states)

## 2. 가짜 배로 시험하기

`sim.py`가 정답을 아는 가짜 배를 만든다.

- 12노트로 북쪽으로 가다가 **200~380초에 1°/초로 우회전**(180° 돌기), 그 뒤 다시 직진
- **20초마다** AIS 보고. 위치 ±5m, SOG ±0.2kn, COG ±2° 잡음

앱은 보고를 받을 때마다 그 값으로 다음 보고 전까지 배를 밀어 그린다. 이 *민 위치*가 실제와 몇 m 틀리는지 직진·선회 구간별로 잰다.

In [ ]:
rep = sim.make_reports()
kf = kalman_cv(rep)

def predict_kf(k, tau):
    e, n, ve, vn = kf[k]
    return e + ve * tau, n + vn * tau

def predict_dr(k, tau):
    """보고 k 의 원래 SOG·COG로 tau 초 곧게 민 위치."""
    v, c = rep["sog"][k] * KN, np.radians(rep["cog"][k])
    return rep["e"][k] + v * np.sin(c) * tau, rep["n"][k] + v * np.cos(c) * tau

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, (xl, yl), title in zip(axes, sim.ZOOM_VIEWS, ["직진 구간 (가로를 늘려 봄)", "선회 구간"]):
    sim.plot_truth(ax, rep)
    sim.plot_extrapolation(ax, rep, predict_dr, "직선 DR (응용 1)", "C3")
    sim.plot_extrapolation(ax, rep, predict_kf, "칼만 필터 + 직선", "C0")
    ax.set_xlim(*xl); ax.set_ylim(*yl); ax.set_title(title); ax.legend(loc="lower right")
axes[0].set_aspect("auto"); axes[1].legend(loc="lower center")
plt.show()

for name, f in [("직선 DR (응용 1)", predict_dr), ("칼만 필터 + 직선", predict_kf)]:
    r = sim.evaluate(rep, f)
    print(f"{name:14s} 직진 {r['직진']:5.2f} m   선회 {r['선회']:5.2f} m")

## 3. 결과

- **직진**에서는 오차가 줄었다. 흔들리는 보고 여러 개를 섞어서 위치·속도가 덜 흔들린다.
- **선회**에서는 거의 그대로다. 모델이 등속 직선이라, 다듬은 값으로도 곧게만 민다.

→ 도는 것을 모델에 넣으려면 **확장 칼만 필터(응용 3)**

## 직접 해 보기

1. `kalman_cv(rep, q=...)`의 `q`를 `0.001`, `1.0`으로 바꿔 보자. 직진·선회 오차는 각각 어떻게 되나? 왜 그런가?

In [ ]:
# 여기서 직접 바꿔 보세요
